# DenseUAV: tối ưu baseline Global (CE + HardMining triplet), Kaggle GPU T4×2

**Baseline**, giống `baselinetest03.ipynb`:
- ViT-S/16-224, head `SingleBranch` với pool `global` (CLS token) → 512-d.
- Loss CE + `HardMiningTripletLoss` (margin 0.3), không KL.
- SGD lr 0.003 (backbone ×0.3), bs 16, 120 epoch, LR ×0.1 ở epoch 70/110.
- Augmentation: random-rotate cho UAV, random-affine + random-erasing cho satellite.

Lần chạy trước đạt **R@1 79.45 / SDM@1 83.92**; paper (Table VII) báo 80.18 / 84.39.

## Các tối ưu áp dụng (O1–O6)
| | Tối ưu | Cải tiến (văn bản 4) | Lúc | Áp dụng cụ thể | Căn cứ |
|---|---|---|---|---|---|
| O1 | **RS head**: descriptor tích phân ring×sector | 1 | train | Mỗi thành phần là trung bình có trọng số $w_i=\int_{U_i} f\,g\,dm\,/\int_{U_i} f\,dm$ (Def. 5.2) của trường token ViT $g$ (hằng trên từng patch). Cụ thể: <br>• $U_i$ là các ô ring×sector của $B(c,R)$, với $R$ = nửa đường chéo ảnh; ảnh đóng vai tập nhìn thấy $V$.<br>• $f=\max(0,1-\lvert x\rvert/R)$, như Ex. 4.1.<br>• Tích phân được tính đúng bằng một ma trận cố định (tổng Riemann 16×16 điểm/patch).<br>• Vì không biết heading UAV, mỗi ring dùng [trung bình, $\lvert\mathrm{DFT}_{1..2}\rvert$ theo trục sector]. Descriptor này bất biến khi xoay $360°/N_\theta$ và khi lật, đúng hướng "rotation invariance" ở mục Future work của bài.<br>• Cấu hình: 2 ring × 8 sector; 1 nhánh CLS + 2 nhánh ring → 1536-d, bằng LPN-B2. | Bài shape-vector (Def. 3.3, 5.2, Thm 5.4, Ex. 4.1, §6). LPN là trường hợp riêng (ring vuông, $f\equiv1$, 1 sector) và đạt +2.9 R@1 trên DenseUAV (paper Table VI). Research không tìm thấy công trình nào dùng ring×sector có trọng số tích phân cho UAV retrieval. |
| O2 | **GeoLS**: nhãn CE làm mềm theo GPS | 2 | train | Target $T=(1-\varepsilon)I+\varepsilon K$, với $\varepsilon=0.1$. $K$ là kernel Gauss($d$, σ=20 m) trên các điểm train khác nằm trong 3σ, chuẩn hoá theo hàng. Đây là ràng buộc liên tục cục bộ dạng *mềm*, không phải Lipschitz cứng. | CEUSP (arXiv 2502.11408): coi hàng xóm GPS là hard-negative thì R@1 còn 52.72, so với 85.84 khi lấy mẫu ngẫu nhiên. Vậy hàng xóm 20 m là cặp gần-dương. G2CL (arXiv 2502.16164) đánh trọng số theo khoảng cách địa lý và được +4.4 R@1 (protocol khác). |
| O3 | **Rot-TTA** cho query | (1) | test | Lấy trung bình embedding query trên 4 góc (0/90/180/270°) × lật ngang. Repo gốc chỉ lật. | Không biết heading UAV; random-rotate lúc train cho +8.8 R@1 (Table IV). Chưa có bằng chứng về TTA trên DenseUAV, nên notebook đo trực tiếp. |
| O4 | **GeoSmooth**: chấm điểm lại theo lân cận GPS của gallery | 2 | test | $s'(q,g)=\sum_{c'}K(c(g),c')\,\bar s(q,c')$, trong đó:<br>• $\bar s$ là điểm trung bình của 6 ảnh vệ tinh (2 năm × 3 scale) cùng vị trí.<br>• $K$ là kernel Gauss với σ=10 m, cố định trước (bằng nửa bước lấy mẫu, nên mỗi vị trí giữ ~60% trọng số của chính nó).<br>• σ=0 nghĩa là chỉ gộp 6 ảnh của cùng vị trí (*loc-fuse*).<br>Hợp lệ vì CSDL vệ tinh có toạ độ, còn query không dùng GPS. | Ý tưởng của nhóm, chưa có bằng chứng ngoài. Bảng sweep σ chỉ dùng để phân tích. |
| O5 | **Framework chẩn đoán lỗi** | 3 | phân tích | • Top-1 của mỗi query được xếp vào correct / near (≤30 m, tức điểm kề) / intra-site / inter-site. Site là thành phần liên thông của đồ thị bán kính 100 m trên GPS.<br>• Bảng chuyển lỗi baseline→best; R@1 theo độ cao.<br>• Trên prototype vị trí của gallery: số cặp *mơ hồ* ($d_{geo}>100$ m nhưng cos ≥ median cos(query, vị trí đúng)), số cặp *gián đoạn* ($d\le30$ m nhưng cos ≤ median của các cặp 60–100 m), Spearman($d_{geo}$, cos), và hằng số Lipschitz thực nghiệm $\lVert\Delta P\rVert/d$. | Văn bản 2–3 (inter/intra-campus, ambiguous spatial pairs). |
| O6 | **Hạ tầng** | – | – | Cache ảnh (cạnh ngắn 384, JPEG q95) vào `/tmp`; 2 run song song, mỗi run một T4; smoke test 1 epoch; dừng an toàn ở 11.3 h. | Run trước mất ~49 s/epoch khi đọc ảnh 1440×1080 từ `/kaggle/input` với 4 vCPU. 3 run tuần tự hết 5.1 h. |

Các tối ưu đã được chứng minh trong literature chỉ dùng làm tham chiếu hoặc để stack, không phải đóng góp mới:
- LPN-B2: +2.9 R@1.
- KL mutual learning: +1.8 R@1 (Table VII).
- Init DINOv2 ViT-S/14: SkyPart (arXiv 2605.11654) đạt 91.85 R@1 nhưng với cả một stack nặng; DINOv2-B thuần chỉ đạt 86.27.

Không đưa vào notebook:
- ViT-B: 87.82 R@1 nhưng tốn ~4× compute, không vừa 12 h cùng các ablation.
- InfoNCE kiểu Sample4Geo: cần bs 128, và kết quả trên DenseUAV dao động 49–81 R@1 tuỳ người train.
- Geo hard-negative sampling: CEUSP cho thấy nó làm kết quả tệ đi.

SOTA đã công bố trên DenseUAV (protocol mỗi bài một khác, chỉ để tham khảo): CAMP 88.72, CEUSP 89.45 (SDM@1 91.01), MEAN 90.18 (ConvNeXt-T), SkyPart 91.85 / 93.59 (DINOv2-S, 27M params).

## Kế hoạch chạy
Thứ tự ưu tiên giảm dần. Mỗi cặp run chạy song song, mỗi run một GPU.

| Run | Head | Loss | lr | Cô lập điều gì |
|---|---|---|---|---|
| `Global` | SingleBranch / global | CE + HM | 0.003 | baseline chạy lại trong cùng pipeline (đối chứng) |
| `Global_GeoLS` | 〃 | GeoLS + HM | 0.003 | O2 |
| `RS_B2` | RS: 2 ring × 8 sector, f decay | CE + HM | 0.01 | O1 |
| `RS_B2_GeoLS` | 〃 | GeoLS + HM | 0.01 | O1 + O2 |
| `LPN_B2` | LPN, block 2 | CE + HM | 0.01 | ring vuông, cùng pipeline (so với O1) |
| `RS_B2_ringonly` | RS: 2 ring × 1 sector | CE + HM | 0.01 | ablation: sector + \|DFT\| có giúp không |
| `RS_B2_GeoLS_KL` | RS | GeoLS + HM + KL | 0.01 | stack với mutual learning của paper |
| `DINOv2S_RS_B2_GeoLS` | RS, backbone DINOv2-S/14 | GeoLS + HM | 0.003 | init mạnh hơn (tuỳ chọn) |

lr 0.01 cho các head chia phần lấy theo paper §VI-E. O3 và O4 được đánh giá trên mọi run mà không cần train lại.

## Dataset khác (kết quả research)
| Dataset | Đặc điểm | Kaggle | Dùng được? |
|---|---|---|---|
| University-1652 | Drone tổng hợp (Google Earth) ở 121–256 m, 1 ảnh vệ tinh/toà nhà; phải xin qua form | `hanahariri/university-1652` (74 MB, chưa xác minh có phải bản đầy đủ) | Không dùng để pretrain: train trên U-1652 rồi test trên DenseUAV chỉ đạt 25.77 R@1 (Table III). Chỉ hợp để test khả năng khái quát. |
| SUES-200 | 200 vị trí, độ cao 150–300 m | không có | Test khái quát sau khi train trên DenseUAV; độ cao quá lớn để pretrain. |
| GTA-UAV | Ảnh game tổng hợp, 80–650 m; HF `Yux1ang/GTA-UAV-LR` (12.8 GB) | không có | Yếu: gap miền lớn. |
| UAV-VisLoc | 400–2000 m, 11 bản đồ 0.3 m | `hailong1610/uav-visloc-dataset` | Chỉ để test khái quát; độ cao gấp 4–20 lần DenseUAV. |
| AnyVisLoc (2025) | 18k ảnh UAV, 15 thành phố, 30–300 m | không có | Ứng viên tốt nhất để test khái quát. |
| VPAIR / AerialVL / ALTO / CVUSA / VIGOR | Độ cao >300 m hoặc ground↔aerial | một phần | Không phù hợp. |
| UL14 | Sinh ra từ DenseUAV | không có | Không độc lập. |

Kết luận: không dataset nào đáng để pretrain trước DenseUAV. "Pretraining" đáng tin duy nhất là init ImageNet/DINOv2, nên notebook không thêm dataset ngoài.

## Lưu ý khi đọc kết quả
- Mỗi cấu hình chỉ chạy 1 seed, nhiễu giữa các lần chạy khoảng ±1 R@1 (Global của ta 79.45 so với paper 80.18). Chênh lệch dưới 1 điểm chưa đủ để kết luận; trước khi đưa vào paper, hãy chạy lại 2–3 seed cho run tốt nhất.
- σ của GeoSmooth và cấu hình Rot-TTA được cố định trước. Bảng sweep σ trên test chỉ để phân tích, không dùng để chọn mô hình.
- Cache khiến ảnh bị resample 2 lần, nên chỉ so sánh các số trong notebook này với nhau (run `Global` là đối chứng).
- Settings: Accelerator `GPU T4 x2`, Internet `On`, Input `netvoych1k/denseuav`, rồi chọn *Save & Run All*. Run nào chưa chạy khi hết giờ sẽ bị bỏ qua, và các bảng chỉ gồm những run đã xong.

In [ ]:
import glob, os, time
T0 = time.time()  # Kaggle stops a GPU session at 12 h; the run scheduler budgets against this
ROOT = os.path.dirname(glob.glob('/kaggle/input/**/Dense_GPS_ALL.txt', recursive=True)[0])
for d, n in [('train/drone', 2256), ('train/satellite', 2256), ('test/query_drone', 777), ('test/gallery_satellite', 3033)]:
    assert len(os.listdir(f'{ROOT}/{d}')) == n, d
print(ROOT, '|', os.cpu_count(), 'vCPUs')
!nvidia-smi -L

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/Dmmm1997/DenseUAV.git && git -C DenseUAV checkout -q b8de187
%cd /kaggle/working/DenseUAV
!pip install -q -r requirments.txt
!touch datasets/__init__.py losses/__init__.py optimizers/__init__.py tool/__init__.py
!sed -i 's/np\.bool8/np.bool_/g; s/np\.in1d/np.isin/g' evaluateDistance.py evaluate_gpu.py
!sed -i 's/timm\.models\.layers/timm.layers/' models/Backbone/cvt.py
!mkdir -p checkpoints logs
# fetch pretrained weights once so the two parallel runs do not race on the HF cache
!python -c "import timm; [timm.create_model(m, pretrained=True) for m in ('vit_small_patch16_224', 'vit_small_patch14_dinov2.lvd142m')]"

## Code mới
- `models/Head/RS.py` (O1)
- `losses/geo.py` (O2)
- `rot_tta.py` (O3)
- `evalkit.py`: metric của repo viết lại dạng vector, kèm O4 và O5

Mỗi file có self-check trong `__main__`; cell vá repo sẽ chạy các self-check này.

In [ ]:
%%writefile models/Head/RS.py
import math
import torch
import torch.nn as nn
from .utils import ClassBlock


def ring_sector_operator(G, n_rings, n_sectors, n_freqs, weight="decay", ss=16):
    """Linear maps (real, imag), each (n_rings*(n_freqs+1), G*G), from the G x G patch-token field to the
    DFT over sectors of the cell averages  w_i = int_{U_i} f g dm / int_{U_i} f dm  (Def. 5.2 of the shape-vector paper).
    Cells U_i: n_rings equal-width rings x n_sectors sectors of B(c, R), c = image centre, R = half-diagonal,
    so the image square is the visible set. f = max(0, 1-|x|/R) ('decay') or 1 ('uniform').
    Integrals are Riemann sums on ss x ss sub-samples per patch (exact for the patch-constant field up to O(1/ss))."""
    t = (torch.arange(G * ss, dtype=torch.float64) + 0.5) / ss - G / 2
    y, x = torch.meshgrid(t, t, indexing="ij")
    R = G / math.sqrt(2)
    r = torch.sqrt(x ** 2 + y ** 2)
    f = (1 - r / R).clamp(min=0) if weight == "decay" else torch.ones_like(r)
    ring = (r / R * n_rings).long().clamp(max=n_rings - 1)
    # sectors centred on the axes: boundaries at irrational slopes, so no sub-sample lies on one
    ang = (torch.atan2(y, x) + math.pi / n_sectors) % (2 * math.pi)
    sec = (ang / (2 * math.pi) * n_sectors).long().clamp(max=n_sectors - 1)
    idx = torch.arange(G * ss) // ss
    patch = idx[:, None] * G + idx[None, :]
    A = torch.zeros(n_rings * n_sectors, G * G, dtype=torch.float64)
    A.index_put_(((ring * n_sectors + sec).flatten(), patch.flatten()), f.flatten(), accumulate=True)
    assert (A.sum(1) > 0).all(), "empty ring-sector cell"
    A = (A / A.sum(1, keepdim=True)).view(n_rings, n_sectors, G * G)
    jk = torch.arange(n_sectors, dtype=torch.float64)[:, None] * torch.arange(n_freqs + 1)[None] * 2 * math.pi / n_sectors
    re = torch.einsum("rjp,jk->rkp", A, torch.cos(jk) / n_sectors).flatten(0, 1)
    im = torch.einsum("rjp,jk->rkp", A, -torch.sin(jk) / n_sectors).flatten(0, 1)
    return re.float(), im.float()


class RS(nn.Module):
    """Global CLS branch + one branch per ring. A ring's descriptor is [mean over its sectors, |DFT_1..K| over sectors]:
    shifting sectors (rotating the UAV heading by 360/n_sectors) leaves it unchanged."""

    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        self.block = opt.block
        self.nf = opt.rs_freqs if opt.rs_sectors > 1 else 0
        assert self.nf <= opt.rs_sectors // 2
        self.global_classifier = ClassBlock(opt.in_planes, opt.nclasses, opt.droprate, num_bottleneck=opt.num_bottleneck)
        self.ring_classifiers = nn.ModuleList(
            ClassBlock(opt.in_planes * (self.nf + 1), opt.nclasses, opt.droprate, num_bottleneck=opt.num_bottleneck)
            for _ in range(opt.block))
        self.ops = {}

    def pool(self, tokens):  # (B, G*G, C) -> (B, rings, (nf+1)*C)
        key = (tokens.size(1), tokens.device)
        if key not in self.ops:
            G = math.isqrt(tokens.size(1))
            ops = ring_sector_operator(G, self.block, self.opt.rs_sectors, self.nf, self.opt.rs_weight)
            self.ops[key] = [m.to(tokens.device) for m in ops]
        re_op, im_op = self.ops[key]
        with torch.autocast(tokens.device.type, enabled=False):
            t = tokens.float()
            re = torch.einsum("mp,bpc->bmc", re_op, t).unflatten(1, (self.block, self.nf + 1))
            im = torch.einsum("mp,bpc->bmc", im_op, t).unflatten(1, (self.block, self.nf + 1))
            mag = torch.sqrt(re[:, :, 1:] ** 2 + im[:, :, 1:] ** 2 + 1e-8)
            return torch.cat([re[:, :, :1], mag], 2).flatten(2)

    def forward(self, features):
        global_cls, global_feature = self.global_classifier(features[:, 0])
        parts = self.pool(features[:, 1:])
        outs = [c(parts[:, i]) for i, c in enumerate(self.ring_classifiers)]
        total_cls = [global_cls] + [o[0] for o in outs]
        total_features = [global_feature] + [o[1] for o in outs]
        if not self.training:
            total_features = torch.stack(total_features, dim=-1)
        return [total_cls, total_features]


if __name__ == "__main__":  # python -m models.Head.RS
    from types import SimpleNamespace
    for G in (14, 16):
        for S, K, w in ((8, 2, "decay"), (8, 4, "uniform"), (1, 0, "decay")):
            opt = SimpleNamespace(block=2, rs_sectors=S, rs_freqs=K, rs_weight=w, in_planes=8, nclasses=5, droprate=0.5, num_bottleneck=16)
            head = RS(opt).eval()
            tok = torch.randn(3, G * G, 8)
            grid = tok.view(3, G, G, 8)
            for k in (1, 2, 3):  # 90-degree rotations shift sectors by S/4: descriptor must not change
                rot = torch.rot90(grid, k, dims=(1, 2)).reshape(3, G * G, 8)
                assert torch.allclose(head.pool(tok), head.pool(rot), atol=1e-5), (G, S, K, w, k)
            flip = torch.flip(grid, dims=(2,)).reshape(3, G * G, 8)  # mirror image: |DFT| is reflection-invariant too
            assert torch.allclose(head.pool(tok), head.pool(flip), atol=1e-5), (G, S, K, w)
            const = torch.ones(1, G * G, 8)  # constant field: cell averages are 1, angular DFT vanishes
            d = head.pool(const).view(1, 2, K + 1, 8)
            assert torch.allclose(d[:, :, 0], torch.ones(1)) and (d[:, :, 1:] < 1e-3).all()
            out = head(torch.randn(4, G * G + 1, 8))
            assert len(out[0]) == 3 and out[1].shape == (4, 16, 3)
    print("RS head self-check OK")

In [ ]:
%%writefile losses/geo.py
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def load_gps(gps_file):
    """{folder name: (lon, lat)}, parsed exactly like evaluateDistance.py."""
    gps = {}
    with open(gps_file) as fh:
        for line in fh:
            s = line.split(" ")
            gps[s[0].split("/")[-2]] = (float(s[1].split("E")[-1]), float(s[2].split("N")[-1]))
    return gps


def to_meters(lonlat):
    """Local equirectangular projection, metres (error < 0.1% over a campus)."""
    ll = np.asarray(lonlat, dtype=np.float64)
    return np.stack([ll[:, 0] * 111320.0 * np.cos(np.deg2rad(ll[:, 1].mean())), ll[:, 1] * 110574.0], 1)


def geo_soft_targets(xy, eps, sigma):
    """T = (1-eps) I + eps K, K = Gaussian(d, sigma) over the *other* classes within 3 sigma, rows normalised.
    Points with no neighbour keep a hard label."""
    d2 = ((xy[:, None] - xy[None]) ** 2).sum(-1)
    K = np.exp(-d2 / (2 * sigma ** 2)) * (d2 <= (3 * sigma) ** 2)
    np.fill_diagonal(K, 0)
    s = K.sum(1, keepdims=True)
    T = (1 - eps) * np.eye(len(xy)) + eps * K / np.maximum(s, 1e-12)
    lone = np.where(s[:, 0] == 0)[0]
    T[lone, lone] = 1.0
    return torch.tensor(T, dtype=torch.float32)


class GeoSoftCE(nn.Module):
    """Cross-entropy against geo-smoothed labels: the classifier is asked to give neighbouring sampling points
    (20 m apart) a little probability, a soft local-consistency prior instead of a hard Lipschitz constraint."""

    def __init__(self, data_dir, eps=0.1, sigma=20.0):
        super().__init__()
        names = sorted(os.listdir(os.path.join(data_dir, "satellite")))  # label order of Dataloader_University
        gps = load_gps(os.path.join(os.path.dirname(os.path.normpath(data_dir)), "Dense_GPS_ALL.txt"))
        self.register_buffer("T", geo_soft_targets(to_meters([gps[n] for n in names]), eps, sigma))

    def forward(self, logits, labels):
        if self.T.device != logits.device:
            self.T = self.T.to(logits.device)
        return -(self.T[labels] * F.log_softmax(logits.float(), 1)).sum(1).mean()


if __name__ == "__main__":  # python -m losses.geo
    g = np.stack(np.meshgrid(np.arange(5) * 20.0, np.arange(5) * 20.0), -1).reshape(-1, 2)
    xy = np.concatenate([g, [[5000.0, 5000.0]]])  # 5x5 grid at 20 m + one isolated point
    T = geo_soft_targets(xy, 0.1, 20.0).numpy()
    assert np.allclose(T.sum(1), 1) and np.allclose(np.diag(T)[:-1], 0.9) and T[-1, -1] == 1
    c = 12  # grid centre: 20 m neighbours > 28 m diagonals > 40 m; corner-to-corner (113 m) gets nothing
    assert T[c, 13] > T[c, 18] > T[c, 14] > 0 and T[0, 24] == 0 and T[c, -1] == 0
    logits, y = torch.randn(4, 26), torch.tensor([0, 12, 25, 3])
    ce = GeoSoftCE.__new__(GeoSoftCE); nn.Module.__init__(ce)
    ce.register_buffer("T", torch.eye(26))
    assert torch.allclose(ce(logits, y), F.cross_entropy(logits, y), atol=1e-6)  # eps=0 is plain CE
    print("GeoSoftCE self-check OK")

In [ ]:
%%writefile rot_tta.py
# Query features averaged over 4 rotations x 2 flips (UAV heading is unknown). Same order and per-part
# normalisation as test.py, so they can replace query_f of pytorch_result_1.mat directly.
import argparse, os, yaml
import numpy as np
import torch
from types import SimpleNamespace
from torchvision import datasets, transforms
from models.taskflow import make_model

p = argparse.ArgumentParser()
p.add_argument("--name")
p.add_argument("--test_dir")
p.add_argument("--num_worker", type=int, default=2)
a = p.parse_args()
d = f"checkpoints/{a.name}"
opt = SimpleNamespace(**yaml.safe_load(open(f"{d}/opts.yaml")))
model = make_model(opt)
model.load_state_dict(torch.load(f"{d}/net_119.pth", map_location="cpu"))
model = model.cuda().eval()
n_inf = sum(v.numel() for k, v in model.named_parameters() if not k.endswith(("classifier.0.weight", "classifier.0.bias")))

tf = transforms.Compose([transforms.Resize((opt.h, opt.w), interpolation=3), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
ds = datasets.ImageFolder(os.path.join(a.test_dir, "query_drone"), tf)
feats = []
with torch.no_grad():
    for img, _ in torch.utils.data.DataLoader(ds, batch_size=128, num_workers=a.num_worker):
        img, ff = img.cuda(), 0
        for k in range(4):
            r = torch.rot90(img, k, (2, 3))
            ff = ff + model(None, r)[1][1] + model(None, r.flip(3))[1][1]
        if ff.dim() == 3:
            ff = (ff / (ff.norm(dim=1, keepdim=True) * np.sqrt(opt.block))).flatten(1)
        else:
            ff = ff / ff.norm(dim=1, keepdim=True)
        feats.append(ff.cpu())
np.savez(f"{d}/rot_tta.npz", qf=torch.cat(feats).numpy(), paths=np.array([s[0] for s in ds.samples]), params_inf=n_inf)
print(f"rot-TTA query features {tuple(torch.cat(feats).shape)}, inference params {n_inf / 1e6:.2f}M")

In [ ]:
%%writefile evalkit.py
import math
import numpy as np
import scipy.sparse as sp


def metrics(S, ql, gl, q_ll, g_ll):
    """Repo metrics from a query x gallery score matrix: Recall@1/5/10, AP (evaluate_gpu.py),
    SDM@1/3/5/10 (evaluateDistance.py, distances in degrees, s=5e3) and MA@20m (top-1 within 20 m)."""
    order = np.argsort(-S, axis=1)
    good = gl[order] == ql[:, None]
    first = good.argmax(1)
    ap = 0.0
    for row in good:
        rg = np.flatnonzero(row)
        i = np.arange(len(rg))
        prec, old = (i + 1) / (rg + 1), np.where(rg != 0, i / np.maximum(rg, 1), 1.0)
        ap += ((old + prec) / 2).sum() / len(rg)
    top = order[:, :10]
    e = np.exp(-5e3 * np.sqrt(((g_ll[top] - q_ll[:, None]) ** 2).sum(-1)))
    out = {f"R@{k}": 100 * (first < k).mean() for k in (1, 5, 10)}
    out["AP"] = 100 * ap / len(S)
    for K in (1, 3, 5, 10):
        w = 1 - np.arange(K) / K
        out[f"SDM@{K}"] = 100 * ((e[:, :K] * w).sum(1) / w.sum()).mean()
    out["MA@20m"] = 100 * (meters(q_ll, g_ll[order[:, 0]]) < 20).mean()
    return out


def meters(a, b):
    """latlog2meter of tool/evaluateDistance.py, vectorised; a, b are (lon, lat) arrays."""
    la, lb = np.deg2rad(a[:, 1]), np.deg2rad(b[:, 1])
    h = np.sin((la - lb) / 2) ** 2 + np.cos(la) * np.cos(lb) * np.sin(np.deg2rad(a[:, 0] - b[:, 0]) / 2) ** 2
    return 2 * 6378137.0 * np.arcsin(np.sqrt(h))


def geo_smooth(S, gl, g_xy, sigma):
    """Spatial-consistency re-scoring: s'(q,g) = sum_c' K(c(g),c') * mean_{g' in c'} s(q,g'), K Gaussian in metres,
    rows normalised. sigma=0 only fuses the 6 satellite images of each location (scales x years)."""
    _, first, inv = np.unique(gl, return_index=True, return_inverse=True)
    M = sp.csr_matrix((np.ones(len(gl)), (np.arange(len(gl)), inv)))
    C = (M.T @ S.T).T / np.bincount(inv)  # (Q, n_cls) class-mean scores
    if sigma > 0:
        xy = g_xy[first]
        d2 = ((xy[:, None] - xy[None]) ** 2).sum(-1)
        K = np.exp(-d2 / (2 * sigma ** 2)) * (d2 <= (3 * sigma) ** 2)
        C = C @ (K / K.sum(1, keepdims=True)).T
    return C[:, inv]


def sites(xy, radius=100.0):
    """Connected components of the radius graph over sampling points (20 m grid): one id per contiguous site."""
    from scipy.spatial import cKDTree
    from scipy.sparse.csgraph import connected_components
    pairs = cKDTree(xy).query_pairs(radius, output_type="ndarray")
    A = sp.coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(xy), len(xy)))
    return connected_components(A, directed=False)[1]


def error_types(t1, ql, gl, q_xy, g_xy, q_site, g_site, near=30.0):
    """Top-1 outcome per query (t1 = top-1 gallery index): correct / near (wrong point <= 30 m, an adjacent
    sampling point) / intra-site (same contiguous campus area, > 30 m) / inter-site (another area)."""
    d = np.sqrt(((q_xy - g_xy[t1]) ** 2).sum(1))
    return np.where(gl[t1] == ql, "correct", np.where(d <= near, "near", np.where(g_site[t1] == q_site, "intra-site", "inter-site")))


if __name__ == "__main__":
    # cross-check against the repo's own per-query code (compute_mAP of evaluate_gpu.py, evaluateSingle of evaluateDistance.py)
    rng = np.random.default_rng(0)
    nq, nc = 40, 30
    gl = np.repeat(np.arange(nc), 6)
    ql = rng.integers(0, nc, nq)
    ll = 120 + rng.random((nc, 2)) * 1e-3
    g_ll, q_ll = ll[gl], ll[ql]
    S = rng.standard_normal((nq, len(gl)))
    m = metrics(S, ql, gl, q_ll, g_ll)

    def compute_mAP(index, good_index):
        ap, cmc = 0, np.zeros(len(index), int)
        rows_good = np.flatnonzero(np.isin(index, good_index))
        cmc[rows_good[0]:] = 1
        for i in range(len(good_index)):
            precision = (i + 1) * 1.0 / (rows_good[i] + 1)
            old_precision = i * 1.0 / rows_good[i] if rows_good[i] != 0 else 1.0
            ap += 1.0 / len(good_index) * (old_precision + precision) / 2
        return ap, cmc

    def evaluateSingle(distance, K):
        weight = np.ones(K) - np.array(range(0, K, 1)) / K
        return np.sum(1 / np.exp(distance * 5e3) * weight) / np.sum(weight)

    CMC, AP, sdm = 0, 0, {K: 0 for K in (1, 3, 5, 10)}
    for i in range(nq):
        index = np.argsort(S[i])[::-1]
        ap, cmc = compute_mAP(index, np.argwhere(gl == ql[i]))
        CMC, AP = CMC + cmc, AP + ap
        for K in sdm:
            sdm[K] += evaluateSingle(np.sqrt(((g_ll[index[:K]] - q_ll[i]) ** 2).sum(1)), K) / nq
    ref = {"R@1": CMC[0], "R@5": CMC[4], "R@10": CMC[9]}
    for k, v in ref.items():
        assert abs(m[k] - 100 * v / nq) < 1e-9, k
    assert abs(m["AP"] - 100 * AP / nq) < 1e-9
    for K in sdm:
        assert abs(m[f"SDM@{K}"] - 100 * sdm[K]) < 1e-9, K
    # geo_smooth: sigma=0 equals averaging each class's 6 scores; sigma>0 keeps the class structure
    S0 = geo_smooth(S, gl, ll[gl] * 1e5, 0)
    assert np.allclose(S0[:, :6], S[:, :6].mean(1, keepdims=True))
    assert geo_smooth(S, gl, ll[gl] * 1e5, 20.0).shape == S.shape
    xy = np.concatenate([np.stack(np.meshgrid(np.arange(4) * 20.0, np.arange(4) * 20.0), -1).reshape(-1, 2), [[1e4, 0]]])
    st = sites(xy)
    assert len(set(st)) == 2
    et = error_types(np.array([0, 1, 10, 16]), np.zeros(4, int), np.arange(17), np.zeros((4, 2)), xy, np.full(4, st[0]), st)
    assert list(et) == ["correct", "near", "intra-site", "inter-site"]
    print("evalkit self-check OK (random scores, chance-level numbers expected):", "  ".join(f"{k} {v:.2f}" for k, v in m.items()))

In [ ]:
def patch(path, old, new):
    s = open(path).read()
    if new not in s:
        assert old in s, (path, old)
        open(path, 'w').write(s.replace(old, new, 1))

patch('train.py', "    parser.add_argument('--head_pool',",
      "    parser.add_argument('--geo_ls', default=0.0, type=float, help='GeoLS mass on GPS neighbours, 0 = plain CE')\n"
      "    parser.add_argument('--geo_sigma', default=20.0, type=float, help='GeoLS Gaussian sigma in metres')\n"
      "    parser.add_argument('--rs_sectors', default=8, type=int, help='RS head: sectors per ring')\n"
      "    parser.add_argument('--rs_freqs', default=2, type=int, help='RS head: |DFT| harmonics kept over sectors')\n"
      "    parser.add_argument('--rs_weight', default='decay', type=str, help='RS head: weight f, decay or uniform')\n"
      "    parser.add_argument('--head_pool',")
patch('losses/loss.py', 'from .FocalLoss import FocalLoss', 'from .FocalLoss import FocalLoss\nfrom .geo import GeoSoftCE')
patch('losses/loss.py', '        if opt.cls_loss == "CELoss":\n            self.cls_loss = nn.CrossEntropyLoss()',
      '        if opt.cls_loss == "CELoss" and opt.geo_ls > 0:\n            self.cls_loss = GeoSoftCE(opt.data_dir, opt.geo_ls, opt.geo_sigma)\n'
      '        elif opt.cls_loss == "CELoss":\n            self.cls_loss = nn.CrossEntropyLoss()')
patch('models/Head/head.py', 'from .GeM import GeM', 'from .GeM import GeM\nfrom .RS import RS')
patch('models/Head/head.py', '        elif head == "GeM":', '        elif head == "RS":\n            head_model = RS(opt)\n        elif head == "GeM":')
patch('models/Backbone/backbone.py', '        elif backbone=="ViTS-384":',
      '        elif backbone=="DINOv2S-224":\n'
      '            backbone_model = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, img_size=self.img_size)\n'
      '            output_channel = 384\n'
      '        elif backbone=="ViTS-384":')
!python -m models.Head.RS && python -m losses.geo && python evalkit.py

## O6: cache dữ liệu
Mọi ảnh đều bị resize về 224 khi train/test, nên ta lưu sẵn bản thu nhỏ (cạnh ngắn 384, giữ tỉ lệ, JPEG q95, giữ nguyên tên file) vào `/tmp`. Mỗi epoch không còn phải decode ảnh 1440×1080 từ `/kaggle/input`.
- `RotateAndCrop` vẫn crop trong hình tròn nội tiếp như cũ: hình vuông lấy ra có cạnh ≈271 px, vẫn lớn hơn 224.
- PIL nhận dạng ảnh theo nội dung, nên file `.tif` chứa dữ liệu JPEG vẫn đọc bình thường.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from PIL import Image
import shutil
DATA, SHORT = '/tmp/denseuav', 384

def shrink(src):
    dst = DATA + src[len(ROOT):]
    if os.path.exists(dst):
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    im = Image.open(src)
    im.draft('RGB', (SHORT, SHORT))  # JPEG: DCT-domain downscale, result stays >= SHORT
    im = im.convert('RGB')
    s = SHORT / min(im.size)
    if s < 1:
        im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
    im.save(dst, format='JPEG', quality=95)

t = time.time()
files = [p for d in ('train/drone', 'train/satellite', 'test/query_drone', 'test/gallery_satellite') for p in glob.glob(f'{ROOT}/{d}/*/*')]
with ProcessPoolExecutor(os.cpu_count()) as ex:
    list(ex.map(shrink, files, chunksize=64))
shutil.copy(f'{ROOT}/Dense_GPS_ALL.txt', DATA)
assert sum(len(fs) for _, _, fs in os.walk(DATA)) == len(files) + 1
print(f'{len(files)} images cached in {time.time() - t:.0f}s')
!du -sh {DATA}

## Train
Mỗi run gồm: `train.py` → `test.py` + `evaluate_gpu.py` + `evaluateDistance.py` của repo (cho số chính thức) → `rot_tta.py`.

Mọi run dùng chung `BASE` (protocol của baseline); mỗi dòng trong `RUNS` chỉ ghi phần khác. Hai run liên tiếp chạy song song trên GPU0/GPU1. Log nằm ở `logs/<run>.log`.

In [ ]:
import subprocess
NW = max(1, os.cpu_count() // 2)  # two runs share the vCPUs
BASE = (f'--data_dir {DATA}/train --num_worker {NW} --gpu_ids 0 --backbone ViTS-224 --sample_num 1 --num_bottleneck 512 '
        '--h 224 --w 224 --batchsize 16 --load_from no --ra satellite --re satellite --cj no --rr uav '
        '--cls_loss CELoss --feature_loss HardMiningTripletLoss --head_pool global')
# (name, args overriding BASE), highest priority first
RUNS = [('Global', '--head SingleBranch --block 1 --lr 0.003 --kl_loss no'),
        ('Global_GeoLS', '--head SingleBranch --block 1 --lr 0.003 --kl_loss no --geo_ls 0.1'),
        ('RS_B2', '--head RS --block 2 --lr 0.01 --kl_loss no'),
        ('RS_B2_GeoLS', '--head RS --block 2 --lr 0.01 --kl_loss no --geo_ls 0.1'),
        ('LPN_B2', '--head LPN --block 2 --lr 0.01 --kl_loss no'),
        ('RS_B2_ringonly', '--head RS --block 2 --lr 0.01 --kl_loss no --rs_sectors 1'),
        ('RS_B2_GeoLS_KL', '--head RS --block 2 --lr 0.01 --kl_loss KLLoss --geo_ls 0.1'),
        ('DINOv2S_RS_B2_GeoLS', '--backbone DINOv2S-224 --head RS --block 2 --lr 0.003 --kl_loss no --geo_ls 0.1')]

def pipeline(name, args):
    return (f'python train.py --name {name} {BASE} {args} && cd checkpoints/{name} '
            f'&& python test.py --name {name} --test_dir {DATA}/test --gpu_ids 0 --num_worker {NW} '
            f'&& python evaluate_gpu.py && python evaluateDistance.py --root_dir {DATA} '
            f'&& cd ../.. && python rot_tta.py --name {name} --test_dir {DATA}/test --num_worker {NW}')

def launch(jobs):  # [(name, shell command)]: job i runs on GPU i; blocks until all finish
    procs = [subprocess.Popen(f'({c}) > logs/{n}.log 2>&1', shell=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)})
             for g, (n, c) in enumerate(jobs)]
    for p, (n, _) in zip(procs, jobs):
        p.wait()
        print(f'== {n}: exit {p.returncode}\n' + open(f'logs/{n}.log').read()[-700:])
    return [p.returncode for p in procs]

In [ ]:
# 1-epoch smoke test of every new code path (RS, GeoLS, KL, DINOv2, opts.yaml reload, rot-TTA) before ~10 h of runs
save = ('python -c "import torch, yaml; from types import SimpleNamespace; from models.taskflow import make_model; '
        "o = SimpleNamespace(**yaml.safe_load(open('checkpoints/{n}/opts.yaml'))); "
        "torch.save(make_model(o).state_dict(), 'checkpoints/{n}/net_119.pth')\"")
smoke = {'smoke_vit': dict(RUNS)['RS_B2_GeoLS_KL'], 'smoke_dino': dict(RUNS)['DINOv2S_RS_B2_GeoLS']}
rc = launch([(n, f'python train.py --name {n} {BASE} {a} --num_epochs 1 && {save.format(n=n)} '
                 f'&& python rot_tta.py --name {n} --test_dir {DATA}/test --num_worker {NW}') for n, a in smoke.items()])
assert rc[0] == 0, 'smoke test failed, see logs/smoke_vit.log'
if rc[1]:
    print('DINOv2 failed the smoke test, dropping that run')
    RUNS = [r for r in RUNS if 'DINOv2' not in r[1]]
!rm -rf checkpoints/smoke_*

In [ ]:
T_LIMIT, pair_s = 11.3 * 3600, 3 * 3600  # pessimistic pair time until the first pair is measured
todo = [r for r in RUNS if not os.path.exists(f'checkpoints/{r[0]}/rot_tta.npz')]
for i in range(0, len(todo), 2):
    if time.time() - T0 + pair_s > T_LIMIT:
        print('12 h budget reached, skipping', [n for n, _ in todo[i:]])
        break
    t = time.time()
    launch([(n, pipeline(n, a)) for n, a in todo[i:i + 2]])
    pair_s = time.time() - t
    print(f'pair took {pair_s / 3600:.2f} h, elapsed {(time.time() - T0) / 3600:.2f} h')

## Kết quả
`repo (flip TTA)` là đúng protocol của repo. Cell này tự `assert` rằng evaluator của ta cho ra cùng con số với `results.txt` và `SDM@K(1,100).json` do script của repo sinh ra. Các dòng còn lại cộng dần O3 (rot-TTA), loc-fuse (σ=0) và O4 (GeoSmooth σ=10 m). `dR@1` / `dSDM@1` là chênh lệch so với `Global` theo protocol repo. Bảng được lưu ở `/kaggle/working/results.csv`.

In [ ]:
import json, re, scipy.io
import numpy as np, pandas as pd
from evalkit import metrics, geo_smooth, sites, error_types
from losses.geo import load_gps, to_meters

gps = load_gps(f'{DATA}/Dense_GPS_ALL.txt')
names = sorted(gps)
XY = dict(zip(names, to_meters([gps[n] for n in names])))
folder = lambda p: os.path.basename(os.path.dirname(p.strip()))

F = {}
for n, _ in RUNS:
    B = f'checkpoints/{n}'
    if os.path.exists(f'{B}/rot_tta.npz'):
        r, t = scipy.io.loadmat(f'{B}/pytorch_result_1.mat'), np.load(f'{B}/rot_tta.npz')
        qp, gp = [folder(p) for p in r['query_path']], [folder(p) for p in r['gallery_path']]
        assert [folder(p) for p in t['paths']] == qp
        F[n] = dict(q=r['query_f'], qr=t['qf'], g=r['gallery_f'], params=float(t['params_inf']) / 1e6)
assert F, 'no finished run'
ql, gl = r['query_label'][0], r['gallery_label'][0]  # same test split and order for every run
QPATH, GPATH = [p.strip() for p in r['query_path']], [p.strip() for p in r['gallery_path']]
q_ll, g_ll = np.array([gps[f] for f in qp]), np.array([gps[f] for f in gp])
q_xy, g_xy = np.array([XY[f] for f in qp]), np.array([XY[f] for f in gp])

SIGMA = 10.0  # fixed a priori: half the 20 m sampling step, a location keeps ~60% of its own smoothed score
GS = f'+rotTTA +GeoSmooth{SIGMA:g}m'
VARIANTS = {'repo (flip TTA)': lambda f: f['q'] @ f['g'].T,
            '+rotTTA': lambda f: f['qr'] @ f['g'].T,
            '+rotTTA +loc-fuse': lambda f: geo_smooth(f['qr'] @ f['g'].T, gl, g_xy, 0),
            GS: lambda f: geo_smooth(f['qr'] @ f['g'].T, gl, g_xy, SIGMA)}
rows, TOP1 = [], {}
for n, f in F.items():
    for v, score in VARIANTS.items():
        S = score(f)
        m = metrics(S, ql, gl, q_ll, g_ll)
        TOP1[n, v] = S.argmax(1)
        if v == 'repo (flip TTA)':  # our evaluator must reproduce the repo scripts (1 query = 0.043 points)
            off = dict(re.findall(r'([\w@]+):([\d.]+)', open(f'checkpoints/{n}/results.txt').read()))
            sdm = json.load(open(f'checkpoints/{n}/SDM@K(1,100).json'))
            for k, o in [('R@1', 'Recall@1'), ('R@5', 'Recall@5'), ('R@10', 'Recall@10'), ('AP', 'AP')]:
                assert abs(m[k] - float(off[o])) < 0.1, (n, k, m[k], off[o])
            assert abs(m['SDM@1'] - 100 * sdm['1']) < 0.1, (n, m['SDM@1'], sdm['1'])
        rows.append({'run': n, 'variant': v, 'Params(M)': f['params'], **m})
res = pd.DataFrame(rows).set_index(['run', 'variant'])
if 'Global' in F:
    ref = res.loc[('Global', 'repo (flip TTA)')]
    res['dR@1'], res['dSDM@1'] = res['R@1'] - ref['R@1'], res['SDM@1'] - ref['SDM@1']
res.round(2).to_csv('/kaggle/working/results.csv')
res.round(2)

Sweep σ của GeoSmooth trên run tốt nhất. Bảng này **chỉ để phân tích** (nó dùng chính tập test), không dùng để chọn σ; σ=10 m đã được cố định từ trước.

In [ ]:
best = res.xs('repo (flip TTA)', level='variant')['R@1'].idxmax()
S = F[best]['qr'] @ F[best]['g'].T
sweep = {s: metrics(geo_smooth(S, gl, g_xy, s), ql, gl, q_ll, g_ll) for s in (0, 5, 10, 15, 20, 30)}
pd.DataFrame(sweep).T[['R@1', 'AP', 'SDM@1', 'SDM@5', 'MA@20m']].round(2).rename_axis(f'sigma (m) | {best} +rotTTA')

## O5: chẩn đoán lỗi (cải tiến 3)
1. Phân loại top-1 của mỗi query, cho từng run và từng biến thể test-time (lưu ở `/kaggle/working/error_types.csv`). Kèm R@1 theo độ cao H80/H90/H100.
2. Ma trận chuyển lỗi: baseline (protocol repo) → run tốt nhất (+ rot-TTA + GeoSmooth). Bảng cho biết tối ưu sửa được loại lỗi nào và tạo ra loại lỗi nào.
3. Hình học của embedding trên prototype vị trí (trung bình 6 ảnh vệ tinh), so với GPS:
   - Cặp *mơ hồ*: $d_{geo}\gg0$ nhưng $d_F\ll1$.
   - Cặp *gián đoạn*: $d_{geo}\ll1$ nhưng $d_F\gg0$.
   - Spearman giữa khoảng cách địa lý và độ tương đồng.
   - Hằng số Lipschitz thực nghiệm $\lVert P_i-P_j\rVert/d_{ij}$ trên các cặp kề (median, p95; p95 cao nghĩa là embedding có chỗ nhảy cục bộ).

In [ ]:
import scipy.sparse as sp
from scipy.stats import spearmanr

SITE = dict(zip(names, sites(np.array([XY[n] for n in names]))))
q_site, g_site = np.array([SITE[f] for f in qp]), np.array([SITE[f] for f in gp])
alt = np.array([os.path.basename(p).split('.')[0] for p in QPATH])
print(f'{len(set(SITE.values()))} contiguous sites (100 m radius graph), queries come from {len(set(q_site))} of them')

# 1) top-1 outcome per query, per run and test-time variant
ORDER = ['correct', 'near', 'intra-site', 'inter-site']
ET = {k: error_types(t1, ql, gl, q_xy, g_xy, q_site, g_site) for k, t1 in TOP1.items()}
err = pd.DataFrame({k: pd.Series(v).value_counts(normalize=True) * 100 for k, v in ET.items()}).T.reindex(columns=ORDER).fillna(0)
err.index.names = ['run', 'variant']
err.round(2).to_csv('/kaggle/working/error_types.csv')
display(err.round(2))
display(pd.DataFrame({n: {f'R@1 {h}': 100 * (ET[n, 'repo (flip TTA)'][alt == h] == 'correct').mean()
                          for h in sorted(set(alt))} for n in F}).T.round(2))

# 2) which errors an optimisation removes or creates
best = res.xs('repo (flip TTA)', level='variant')['R@1'].idxmax()
base = 'Global' if 'Global' in F else best
print(f'rows: {base} (repo protocol)   cols: {best} {GS}')
display(pd.crosstab(pd.Categorical(ET[base, 'repo (flip TTA)'], ORDER), pd.Categorical(ET[best, GS], ORDER),
                    rownames=[base], colnames=[best], margins=True))

# 3) geometry of the learned map: gallery location prototypes vs GPS
ucls, first, inv = np.unique(gl, return_index=True, return_inverse=True)
cxy, csite, qi = g_xy[first], g_site[first], np.searchsorted(ucls, ql)
iu = np.triu_indices(len(ucls), 1)
d = np.sqrt(((cxy[iu[0]] - cxy[iu[1]]) ** 2).sum(1))
same = csite[iu[0]] == csite[iu[1]]
adj, mid, far, loc = d <= 30, same & (d > 60) & (d <= 100), d > 100, same & (d <= 200)
M = sp.csr_matrix((np.ones(len(gl)), (inv, np.arange(len(gl)))))

COS, geo = {}, {}
for n, f in F.items():
    P = np.asarray(M @ f['g']); P /= np.linalg.norm(P, axis=1, keepdims=True)
    c = COS[n] = (P @ P.T)[iu]
    q = f['q'] / np.linalg.norm(f['q'], axis=1, keepdims=True)
    tau = np.median((q * P[qi]).sum(1))  # typical query-to-true-location similarity
    lip = np.sqrt(np.clip(2 - 2 * c[adj], 0, None)) / d[adj]  # ||P_i - P_j|| / d_geo, adjacent locations
    geo[n] = {'ambiguous: d>100m & cos>=tau': int((far & (c >= tau)).sum()),
              'discontinuous: d<=30m & cos<=median(60-100m)': int((c[adj] <= np.median(c[mid])).sum()),
              'Spearman(d_geo, cos) same site d<=200m': spearmanr(d[loc], c[loc])[0],
              'Lipschitz median (1/m)': np.median(lip), 'Lipschitz p95 (1/m)': np.percentile(lip, 95)}
pd.DataFrame(geo).T.round(4)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
err.xs('repo (flip TTA)', level='variant').plot.barh(stacked=True, ax=ax[0], color=['#4c9a5f', '#e0b84c', '#d9822b', '#c0392b'])
ax[0].set(xlabel='% of queries', title='top-1 outcome (repo protocol)')
for a, n in zip(ax[1:], [base, best]):
    a.hexbin(d[loc], COS[n][loc], gridsize=40, bins='log')
    a.set(xlabel='geo distance between gallery locations (m)', ylabel='cosine of location prototypes', title=n)
plt.tight_layout(); plt.show()

# the most confident inter-site errors of the best pipeline
S = geo_smooth(F[best]['qr'] @ F[best]['g'].T, gl, g_xy, SIGMA)
t1, bad = S.argmax(1), np.flatnonzero(ET[best, GS] == 'inter-site')
bad = bad[np.argsort(-S[bad, t1[bad]])][:4]
true_g = {c: i for i, c in reversed(list(enumerate(gl)))}
if len(bad):
    fig, ax = plt.subplots(len(bad), 3, figsize=(9, 3 * len(bad)), squeeze=False)
    for row, i in zip(ax, bad):
        for a, p, title in zip(row, [QPATH[i], GPATH[t1[i]], GPATH[true_g[ql[i]]]], ['UAV query', 'top-1 (other site)', 'true location']):
            a.imshow(Image.open(p)); a.set_title(title); a.axis('off')
    plt.tight_layout(); plt.show()